In [19]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)
class Actor(nn.Module):
    def __init__(self,state_dim=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim,64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

Using: cuda


In [20]:
class Critic(nn.Module):
    def __init__(self, state_dim=4):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )

    def forward(self, state):
        return self.net(state).squeeze(-1)

In [21]:
actor = Actor().to(device)
critic = Critic().to(device)

actor_optimizer = optim.Adam(actor.parameters(), lr=3e-4)
critic_optimizer = optim.Adam(critic.parameters(), lr=1e-3)

In [22]:
from torch.distributions import Bernoulli

state = torch.tensor(
    [0.0, 0.0, 0.05, 0.0],
    dtype=torch.float32,
    device=device
)

with torch.no_grad():
    logits = actor(state)
    distribution = Bernoulli(logits=logits)

    action = distribution.sample()
    log_prob = distribution.log_prob(action)
    value = critic(state)

print("Probability of right:", torch.sigmoid(logits).item())
print("Chosen action:", int(action.item()))
print("Log probability:", log_prob.item())
print("Estimated value:", value.item())

Probability of right: 0.434942364692688
Chosen action: 1
Log probability: -0.8325417637825012
Estimated value: 0.09377111494541168


In [23]:
STATE_SCALE = np.array(
    [2.4, 3.0, 0.20944, 3.0],
    dtype=np.float32
)

def normalize_state(state):
    state = np.asarray(state, dtype=np.float32)
    return np.clip(state / STATE_SCALE, -5.0, 5.0)

In [24]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt


# ============================================================
# ENVIRONMENT
# ============================================================

class CartPole:

    def __init__(self):

        self.gravity = 9.8
        self.mass_cart = 1.0
        self.mass_pole = 0.1

        self.total_mass = (
            self.mass_cart + self.mass_pole
        )

        self.length = 0.5
        self.force_mag = 50.0
        self.dt = 0.02

        self.reset()

    def reset(self):

        self.x = 0.0
        self.x_dot = 0.0

        self.theta = 0.05
        self.theta_dot = 0.0

        return self.get_state()

    def get_state(self):

        return [
            self.x,
            self.x_dot,
            self.theta,
            self.theta_dot
        ]

    def is_done(self):

        if abs(self.x) > 2.4:
            return True

        if abs(self.theta) > 12 * math.pi / 180:
            return True

        return False

    def step(self, force):

        force = max(
            -self.force_mag,
            min(self.force_mag, force)
        )

        sin_theta = math.sin(self.theta)
        cos_theta = math.cos(self.theta)

        temp = (
            force
            + self.mass_pole
            * self.length
            * self.theta_dot ** 2
            * sin_theta
        ) / self.total_mass

        theta_acc = (
            self.gravity * sin_theta
            - cos_theta * temp
        ) / (
            self.length *
            (
                4 / 3
                - self.mass_pole
                * cos_theta ** 2
                / self.total_mass
            )
        )

        x_acc = temp - (
            self.mass_pole
            * self.length
            * theta_acc
            * cos_theta
            / self.total_mass
        )

        self.x_dot += self.dt * x_acc
        self.theta_dot += self.dt * theta_acc

        self.x += self.dt * self.x_dot
        self.theta += self.dt * self.theta_dot

        return self.get_state()


# ============================================================
# ACTOR
# ============================================================

def sigmoid(z):
    return 1 / (1 + math.exp(-z))


def policy(state, params):

    x, x_dot, theta, theta_dot = state

    z = (
        params[0] * x
        + params[1] * x_dot
        + params[2] * theta
        + params[3] * theta_dot
    )

    p_right = sigmoid(z)

    if random.random() < p_right:
        action = 1       # RIGHT
    else:
        action = 0       # LEFT

    return action, p_right


# ============================================================
# CRITIC
# ============================================================

def calculate_value(state, params):

    x, x_dot, theta, theta_dot = state

    value = (
        params[0] * x
        + params[1] * x_dot
        + params[2] * theta
        + params[3] * theta_dot
    )

    return value


# ============================================================
# VISUALIZATION
# ============================================================

def draw_cartpole(state):

    x, x_dot, theta, theta_dot = state

    pole_length = 0.5

    pole_x = x + pole_length * math.sin(theta)
    pole_y = pole_length * math.cos(theta)

    plt.clf()

    plt.plot([-2.4, 2.4], [0, 0])

    plt.plot(
        [x - 0.2, x + 0.2],
        [0.1, 0.1],
        linewidth=5
    )

    plt.plot(
        [x, pole_x],
        [0.1, 0.1 + pole_y],
        linewidth=3
    )

    plt.xlim(-2.4, 2.4)
    plt.ylim(-0.1, 1.0)

    plt.pause(0.02)

In [25]:
env = CartPole()

gamma = 0.99
epsilon = 0.2

actor_optimizer = optim.Adam(actor.parameters(), lr=3e-4)
critic_optimizer = optim.Adam(critic.parameters(), lr=1e-3)

In [26]:
states = []
actions = []
rewards = []
old_log_probs = []
values = []

state = env.reset()
done = False
episode_reward = 0

while not done:
    normalized_state = normalize_state(state)

    state_tensor = torch.tensor(
        normalized_state,
        dtype=torch.float32,
        device=device
    )

    with torch.no_grad():
        logits = actor(state_tensor)
        dist = Bernoulli(logits=logits)

        action_tensor = dist.sample()
        log_prob = dist.log_prob(action_tensor)
        value = critic(state_tensor)

    action = int(action_tensor.item())

    force = env.force_mag if action == 1 else -env.force_mag
    next_state = env.step(force)

    done = env.is_done()
    reward = 1.0

    states.append(normalized_state)
    actions.append(action)
    rewards.append(reward)
    old_log_probs.append(log_prob.item())
    values.append(value.item())

    episode_reward += reward
    state = next_state

print("Episode length:", len(rewards))
print("Episode reward:", episode_reward)

Episode length: 4
Episode reward: 4.0


In [27]:
returns = np.zeros(len(rewards), dtype=np.float32)

G = 0.0

for t in reversed(range(len(rewards))):
    G = rewards[t] + gamma * G
    returns[t] = G

print("Rewards:", rewards[:5])
print("Returns:", returns[:5])

Rewards: [1.0, 1.0, 1.0, 1.0]
Returns: [3.940399 2.9701   1.99     1.      ]


In [28]:
advantages = (
    returns - np.asarray(values, dtype=np.float32)
)

print("Advantages:", advantages[:5])

Advantages: [3.8239446 2.8645155 1.8921822 0.8499162]


In [29]:
states_tensor = torch.tensor(
    np.asarray(states),
    dtype=torch.float32,
    device=device
)

actions_tensor = torch.tensor(
    actions,
    dtype=torch.float32,
    device=device
)

old_log_probs_tensor = torch.tensor(
    old_log_probs,
    dtype=torch.float32,
    device=device
)

returns_tensor = torch.tensor(
    returns,
    dtype=torch.float32,
    device=device
)

advantages_tensor = torch.tensor(
    advantages,
    dtype=torch.float32,
    device=device
)

In [31]:
new_logits = actor(states_tensor)
new_dist = Bernoulli(logits=new_logits)

new_log_probs = new_dist.log_prob(actions_tensor)

ratio = torch.exp(
    new_log_probs - old_log_probs_tensor
)

print("First five ratios:", ratio.detach().cpu().numpy())

First five ratios: [1. 1. 1. 1.]
